# 04: Distributions, independent coder vs Opus
Same records, two coders: proportions of P/NC/D/U/DEV (Arm A) and relevance and NC/D (Arm B), with Wilson 95% CIs. If only part of the sample is coded, the Opus column is restricted to the same records.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 80)

In [2]:
from io_utils import load_coder_sheet, load_opus_codes, TABLES
from coding_schema import is_coded, validation_errors, blocking
coder = load_coder_sheet()
coder = coder[coder.apply(is_coded, axis=1)].copy()
assert len(coder), "nothing coded yet"
assert not any(blocking(validation_errors(r)) for _, r in coder.iterrows()), "run 03 and fix validation errors first"
opus = load_opus_codes()
both = coder.merge(opus, on=["arm", "pmid"], suffixes=("_ind", "_opus"))
assert len(both) == len(coder)
print(len(both), "records coded by both;", both.priority_10pct.eq("1").sum(), "of them in the planned 10% subset")

747 records coded by both; 75 of them in the planned 10% subset


In [3]:
from stats_utils import distribution
from coding_schema import CODEBOOK
rows = []
def add(label, sub, field, levels, drop=None):
    for who in ["ind", "opus"]:
        v = sub[f"{field}_{who}"]
        d = distribution(v[v != drop] if drop else v, levels)
        d.insert(0, "coder", {"ind": "independent", "opus": "Opus 5.5"}[who]); d.insert(0, "table", label)
        rows.append(d)
A = both[both.arm == "A"]
add("A: all", A, "trajectory_code", CODEBOOK["trajectory_code"])
add("A: senescence-relevant (each coder's own DEV excluded)", A, "trajectory_code", ["P", "NC", "D", "U"], drop="DEV")
B = both[both.arm == "B"]
add("B: screen", B, "screen_relevant", ["yes", "no"])
add("B: relevant-by-both, NC vs D", B[(B.screen_relevant_ind == "yes") & (B.screen_relevant_opus == "yes")],
    "trajectory_code", ["P", "NC", "D"])
dist = pd.concat(rows, ignore_index=True)
dist["cell"] = dist.apply(lambda r: f"{r['count']}/{r['n']} ({100*r['proportion']:.1f}%; {100*r['ci_low']:.1f}-{100*r['ci_high']:.1f})", axis=1)
dist.to_csv(TABLES / "distributions_independent_vs_opus.tsv", sep="\t", index=False)
dist.pivot_table(index=["table", "level"], columns="coder", values="cell", aggfunc="first", sort=False)

coder                                                                        independent  \
table                                                  level                               
A: all                                                 P      137/350 (39.1%; 34.2-44.3)   
                                                       NC         19/350 (5.4%; 3.5-8.3)   
                                                       D           7/350 (2.0%; 1.0-4.1)   
                                                       U      141/350 (40.3%; 35.3-45.5)   
                                                       DEV     46/350 (13.1%; 10.0-17.1)   
A: senescence-relevant (each coder's own DEV excluded) P      137/304 (45.1%; 39.6-50.7)   
                                                       NC         19/304 (6.2%; 4.0-9.6)   
                                                       D           7/304 (2.3%; 1.1-4.7)   
                                                       U      141/304 (46.4%; 40.9-52.0)   
B: screen                                              yes      45/397 (11.3%; 8.6-14.8)   
                                                       no     352/397 (88.7%; 85.2-91.4)   
B: relevant-by-both, NC vs D                           P         9/34 (26.5%; 14.6-43.1)   
                                                       NC       20/34 (58.8%; 42.2-73.6)   
                                                       D          5/34 (14.7%; 6.4-30.1)   

coder                                                                           Opus 5.5  
table                                                  level                              
A: all                                                 P       84/350 (24.0%; 19.8-28.7)  
                                                       NC         15/350 (4.3%; 2.6-6.9)  
                                                       D           9/350 (2.6%; 1.4-4.8)  
                                                       U      201/350 (57.4%; 52.2-62.5)  
                                                       DEV      41/350 (11.7%; 8.8-15.5)  
A: senescence-relevant (each coder's own DEV excluded) P       84/309 (27.2%; 22.5-32.4)  
                                                       NC         15/309 (4.9%; 3.0-7.9)  
                                                       D           9/309 (2.9%; 1.5-5.4)  
                                                       U      201/309 (65.0%; 59.6-70.2)  
B: screen                                              yes    108/397 (27.2%; 23.1-31.8)  
                                                       no     289/397 (72.8%; 68.2-76.9)  
B: relevant-by-both, NC vs D                           P        11/34 (32.4%; 19.1-49.2)  
                                                       NC       20/34 (58.8%; 42.2-73.6)  
                                                       D           3/34 (8.8%; 3.0-23.0)

## The manuscript's direct claim, per coder
Among D records: individual/observed vs population/inferred (residual combinations reported, not dropped).

In [4]:
out = []
for who in ["ind", "opus"]:
    d = both[both[f"trajectory_code_{who}"] == "D"]
    combo = d[f"scale_{who}"] + " / " + d[f"observation_{who}"]
    out.append(combo.value_counts().rename({"ind": "independent", "opus": "Opus 5.5"}[who]))
claim = pd.concat(out, axis=1).fillna(0).astype(int)
claim.to_csv(TABLES / "D_scale_observation_by_coder.tsv", sep="\t")
claim

,independent,Opus 5.5
/,3,1
population / observed,3,3
population / inferred,3,4
individual / inferred,2,5
individual / observed,2,6
